## 按国家生成病毒蛋白序列PKL数据集

该脚本旨在根据不同国家的采样数据，为每个病毒样本生成包含27种蛋白的序列和结构编码信息，并最终为每个国家生成一个独立的PKL文件。

**核心流程:**
1. **配置设置:** 在代码的第二部分设置所有必要的路径、国家列表、蛋白列表和计算设备。
2. **模型加载:** 加载预训练的ESM-3模型 (`esm3_sm_open_v1`) 至指定GPU。
3. **参考数据预加载:** 为了提高效率，脚本会预先加载并编码所有27种蛋白的**参考序列 (来自`fasta/`)** 和 **参考PDB结构 (来自`pdb/`)**。
4. **全量序列数据预加载:** 将`/data2/zhoukaitao/01evoModel/ViGTK/` 目录下所有蛋白的原始FASTA序列一次性读入内存，并按蛋白类型和`virus_id`进行组织，便于快速查找。
5. **国家数据处理主循环:**
   - 遍历 `TARGET_COUNTRIES` 列表中的每个国家。
   - 加载该国的采样数据CSV文件 (`stage2data_sample/XXX_sampled_dataset.csv`)。
   - 将“采样时间”列转换为以天为单位的数值 `days`。
   - 遍历27种蛋白，从内存中为每个病毒样本提取对应序列，使用ESM-3模型进行编码。
   - 将所有蛋白的编码信息合并到每个`virus_id`的最终记录中。
6. **保存结果:** 将每个国家处理好的数据集（一个包含多个字典的列表）保存为独立的PKL文件。

In [1]:
import os
import pickle
import pandas as pd
import numpy as np
import torch
from datetime import datetime
from tqdm.notebook import tqdm

# 导入ESM相关模块
from esm.models.esm3 import ESM3
from esm.sdk.api import ESMProtein

# 假设ioutils.py在当前目录下或Python路径中
import ioutils 

# 忽略pandas在读取混合类型列时可能产生的警告
import warnings
warnings.filterwarnings('ignore', category=pd.errors.DtypeWarning)

### 1. 配置模块
**请在此处修改所有路径和参数。**
将 `TARGET_COUNTRIES` 列表设置为单个国家（例如 `['USA']`）可以方便地进行调试，调试成功后再设置为 `ALL_COUNTRIES` 以处理所有数据。

In [23]:
# ----- 主要配置 -----

# 1. 指定要使用的GPU设备
DEVICE = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")

# 2. 定义27种蛋白的列表 (统一使用大写，这将在最终的pkl文件中作为key)
# PROTEIN_LIST_UPPER = [
#     'NSP1', 'NSP2', 'NSP3', 'NSP4', 'NSP5', 'NSP6', 'NSP7', 'NSP8', 'NSP9', 
#     'NSP10', 'NSP11', 'NSP12', 'NSP13', 'NSP14', 'NSP15', 'NSP16', 'S', 'ORF3a', 
#     'E', 'M', 'ORF6', 'ORF7a', 'ORF7b', 'ORF8', 'N', 'ORF9b', 'ORF10'
# ]

# review
PROTEIN_LIST_UPPER = [
    'S'
]

# 3. 国家列表
ALL_COUNTRIES = ['USA', 'United Kingdom', 'Germany', 'Canada', 'Denmark', 'Japan', 'France']

# !!! 在此设置要处理的国家，可以只放一个国家进行测试 !!!
TARGET_COUNTRIES = ['Random'] # 示例: ['USA', 'Japan'] 或 ALL_COUNTRIES

# ----- 路径配置 -----

# 4. 采样后的国家数据集CSV文件所在文件夹
SAMPLED_DATA_DIR = '/data2/zhoukaitao/01evoModel/dataset/251020_review/random/sample_list/'

# 5. 包含所有病毒序列的原始FASTA文件的文件夹
RAW_FASTA_DIR = '/data2/zhoukaitao/01evoModel/ViGTK/'

# 6. 参考序列FASTA文件夹 (蛋白名大写)
REF_FASTA_DIR = './fasta/'

# 7. 参考结构PDB文件夹 (蛋白名大写)
REF_PDB_DIR = './pdb/'

# 8. 输出PKL文件的文件夹
OUTPUT_PKL_DIR = '/data2/zhoukaitao/01evoModel/dataset/251020_review/random/ESM3/'
os.makedirs(OUTPUT_PKL_DIR, exist_ok=True)

print(f"将使用设备: {DEVICE}")
print(f"将处理的国家: {TARGET_COUNTRIES}")
print(f"输出目录已确认: {OUTPUT_PKL_DIR}")

将使用设备: cuda:3
将处理的国家: ['Random']
输出目录已确认: /data2/zhoukaitao/01evoModel/dataset/251020_review/random/ESM3/


### 2. 加载ESM-3模型
此步骤会加载模型到指定的GPU设备，仅执行一次。

In [3]:
print("正在加载 ESM-3 模型...")
model = ESM3.from_pretrained("esm3_sm_open_v1", DEVICE)
model.eval() # 设置为评估模式，关闭dropout等
print("ESM-3 模型加载完成。")

正在加载 ESM-3 模型...
ESM-3 模型加载完成。


### 3. 参考数据预加载与编码
为了避免在主循环中重复读取和编码，我们首先一次性处理所有27种蛋白的参考序列和PDB文件。结果存储在 `ref_data` 字典中。

In [4]:
ref_data = {}
print("开始预加载和编码参考数据...")

for pro_name in tqdm(PROTEIN_LIST_UPPER, desc="编码参考蛋白"):
    ref_data[pro_name] = {}
    
    # 1. 加载并编码参考序列 (文件名使用大写)
    ref_fasta_path = os.path.join(REF_FASTA_DIR, f"{pro_name}.fasta")
    _, ref_seq = next(ioutils.readFasta(ref_fasta_path, remove="*toX", truclength=None))
    ref_protein_seq = ESMProtein(sequence=ref_seq)
    with torch.no_grad():
        encoded_ref_seq = model.encode(ref_protein_seq).sequence.cpu().numpy()
    
    ref_data[pro_name]['seq_str'] = ref_seq
    ref_data[pro_name]['encoded_seq'] = encoded_ref_seq
    ref_data[pro_name]['len'] = len(ref_seq)

    # 2. 加载并编码参考PDB结构 (文件名使用大写)
    ref_pdb_path = os.path.join(REF_PDB_DIR, f"{pro_name}.pdb")
    ref_protein_pdb = ESMProtein.from_pdb(ref_pdb_path)
    with torch.no_grad():
        encoded_ref_pdb = model.encode(ref_protein_pdb).structure.cpu().numpy()
    ref_data[pro_name]['encoded_pdb'] = encoded_ref_pdb

print("所有参考数据预加载和编码完成。")

开始预加载和编码参考数据...


编码参考蛋白:   0%|          | 0/1 [00:00<?, ?it/s]

所有参考数据预加载和编码完成。


In [5]:
ref_data

{'S': {'seq_str': 'MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFCNDPFLGVYYHKNNKSWMESEFRVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPINLVRDLPQGFSALEPLVDLPIGINITRFQTLLALHRSYLTPGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLTGTGVLTESNKKFLPFQQFGRDIADTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNQVAVLYQDVNCTEVPVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEHVNNSYECDIPIGAGICASYQTQTNSPRRARSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLNRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKDFGGFNFSQILPDPSKPSKRSFIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGAALQIPFAMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLSSTASALGKLQDVVNQNAQALNTLVKQLSSNFGAISSVLNDIL

### 4. 全量序列数据预加载
将所有原始FASTA序列加载到内存中，构建一个 `protein -> virus_id -> sequence` 结构，方便快速查找。
**注意：** 此处会处理蛋白名称的大小写，以匹配文件名（例如，NSP1 -> nsp1.fasta）。

In [6]:
all_protein_seqs = {pro_name: {} for pro_name in PROTEIN_LIST_UPPER}

print("开始预加载全量序列数据...")
for pro_name_upper in tqdm(PROTEIN_LIST_UPPER, desc="加载全量FASTA"):
    # 根据蛋白名规则转换文件名 (NSP -> nsp)
    if pro_name_upper.startswith('NSP'):
        fasta_filename_pro_name = pro_name_upper.lower()
    else:
        fasta_filename_pro_name = pro_name_upper
    
    raw_fasta_path = os.path.join(RAW_FASTA_DIR, f"202001_20250712_nextclade.cds_translation.{fasta_filename_pro_name}.fasta")
    
    if not os.path.exists(raw_fasta_path):
        print(f"  - 警告: 找不到文件 {raw_fasta_path}，跳过蛋白 {pro_name_upper}")
        continue
    
    # 使用ioutils读取fasta文件
    seqs_generator = ioutils.readFasta(raw_fasta_path, truclength=None, remove="*toX", checkseq=None)
    for header, seq in seqs_generator:
        virus_id = header.strip()
        all_protein_seqs[pro_name_upper][virus_id] = seq

print("全量序列数据加载完成。")

开始预加载全量序列数据...


加载全量FASTA:   0%|          | 0/1 [00:00<?, ?it/s]

全量序列数据加载完成。


### 5. 按国家处理数据并生成PKL文件
遍历 `TARGET_COUNTRIES` 列表，对每个国家执行完整的处理流程。

In [24]:
cnt = 0
for country in TARGET_COUNTRIES:
    print(f"\n{'='*25} 开始处理国家: {country} {'='*25}")
    
    # 1. 加载该国的采样数据CSV并计算days
    sampled_csv_path = os.path.join(SAMPLED_DATA_DIR, f"{country}_sampled_dataset.csv")
    if not os.path.exists(sampled_csv_path):
        print(f"错误: 找不到采样文件 {sampled_csv_path}。跳过该国家。")
        continue
        
    print(f"  - 加载采样数据: {os.path.basename(sampled_csv_path)}")
    df_sampled = pd.read_csv(sampled_csv_path)
    df_sampled['days'] = df_sampled['采样时间'].astype(str).apply(ioutils.getDate())
    
    # 预先构建一个以virus_id为键的字典，用于聚合所有蛋白信息
    virus_data_map = {
        row['virus_id']: {'id': row['virus_id'], 'days': row['days']}
        for _, row in df_sampled.iterrows()
    }
    
    # 2. 遍历27种蛋白，进行编码和数据合并
    for pro_name in tqdm(PROTEIN_LIST_UPPER, desc=f"处理 {country} 的蛋白"):
        
        # 获取当前蛋白的参考数据和全量序列
        current_ref_data = ref_data[pro_name]
        current_protein_sequences = all_protein_seqs[pro_name]
        
        # 遍历该国的所有采样ID
        for virus_id in virus_data_map.keys():
            seq = current_protein_sequences.get(virus_id)
            
            # 过滤无效数据：序列不存在、长度与参考不符、采样日期无效
            if seq is None or len(seq) != current_ref_data['len'] or virus_data_map[virus_id]['days'] < 0:
                print(virus_id, pro_name, virus_data_map[virus_id]['days'], len(seq), seq)
                continue
            
            # 编码序列
            try:
                protein_obj = ESMProtein(sequence=seq)
                with torch.no_grad():
                    encoded_seq = model.encode(protein_obj).sequence.cpu().numpy()
            except Exception as e:
                # print(f"      - 编码失败: virus_id={virus_id}, protein={pro_name}. Error: {e}") # 可选的详细错误日志
                continue

            # 按照要求的格式构建蛋白字典 (key全部使用大写蛋白名)
            pro_dict = {
                "seq_t": encoded_seq,
                "structure_t": current_ref_data['encoded_pdb']
            }
            aligned_pro_dict = {
                "seq_t": current_ref_data['encoded_seq'],
                "structure_t": current_ref_data['encoded_pdb']
            }
            
            # 将蛋白数据添加到该virus_id的记录中
            virus_data_map[virus_id][pro_name] = pro_dict
            virus_data_map[virus_id][f"aligned_{pro_name}"] = aligned_pro_dict
            
            # ESMC数据
            # virus_data_map[virus_id][pro_name] = encoded_seq
            # virus_data_map[virus_id][f"aligned_{pro_name}"] = current_ref_data['encoded_seq']

            
            
    # 3. 过滤掉没有集齐所有27种蛋白信息的样本
    final_results = []
    expected_key_count = 2 + 2 * len(PROTEIN_LIST_UPPER) # id, days, 和 27*2 个蛋白相关key
    for virus_id, data in virus_data_map.items():
        if len(data.keys()) == expected_key_count:
            final_results.append(data)
    
    print(f"  - 初始采样样本数: {len(df_sampled)}")
    print(f"  - 最终有效样本数 (包含全部27种蛋白): {len(final_results)}")

    # 4. 保存结果到PKL文件
    if final_results:
        output_path = os.path.join(OUTPUT_PKL_DIR, f"{country}.pkl")
        with open(output_path, "wb") as f:
            pickle.dump(final_results, f)
        print(f"  - 成功将数据保存到: {output_path}")
    else:
        print("  - 警告: 没有生成任何有效数据，不创建PKL文件。")

print(f"\n{'='*25} 所有任务完成 {'='*25}")


========================= 开始处理国家: Random =========================
  - 加载采样数据: Random_sampled_dataset.csv


处理 Random 的蛋白:   0%|          | 0/1 [00:00<?, ?it/s]

  - 初始采样样本数: 100000
  - 最终有效样本数 (包含全部27种蛋白): 100000
  - 成功将数据保存到: /data2/zhoukaitao/01evoModel/dataset/251020_review/random/ESM3/Random.pkl

========================= 所有任务完成 =========================


### 6. 验证生成的PKL文件
加载PKL文件，检查内容和结构

In [25]:
if TARGET_COUNTRIES:
    country_to_check = TARGET_COUNTRIES[0] # 检查处理的第一个国家
    pkl_file_path = os.path.join(OUTPUT_PKL_DIR, f"{country_to_check}.pkl")

    if os.path.exists(pkl_file_path):
        print(f"正在加载并验证文件: {pkl_file_path}")
        with open(pkl_file_path, 'rb') as f:
            loaded_data = pickle.load(f)
        
        if isinstance(loaded_data, list) and len(loaded_data) > 0:
            print(f"文件加载成功，共包含 {len(loaded_data)} 条记录。")
            first_item = loaded_data[0]
            print("\n--- 第一条记录结构检查 ---")
            print(f"  - ID: {first_item.get('id')}")
            print(f"  - Days: {first_item.get('days')}")
            print(f"  - 包含的键数量: {len(first_item.keys())} (预期为 {2 + 2 * len(PROTEIN_LIST_UPPER)})")
            
            # 检查几个关键蛋白是否存在且key为大写
            key_check_passed = True
            for check_key in ['S', 'NSP1', 'NSP15', 'N']:
                if check_key not in first_item or f'aligned_{check_key}' not in first_item:
                    print(f"  - 错误: 记录中缺少蛋白 '{check_key}' 或 'aligned_{check_key}' 的信息。")
                    key_check_passed = False
            
            if key_check_passed:
                print("  - 关键蛋白键名检查通过 (均为大写)。")
                print(f"    - S['seq_t'] shape: {first_item['S']['seq_t'].shape}")
                print(f"    - S['structure_t'] shape: {first_item['S']['structure_t'].shape}")
                print(f"    - aligned_S['seq_t'] shape: {first_item['aligned_S']['seq_t'].shape}")
        elif isinstance(loaded_data, list) and len(loaded_data) == 0:
             print("文件加载成功，但列表为空，没有有效的样本数据。")
        else:
            print("文件内容不是一个列表或格式不正确。")
    else:
        print(f"错误: 验证文件 {pkl_file_path} 不存在。")

正在加载并验证文件: /data2/zhoukaitao/01evoModel/dataset/251020_review/random/ESM3/Random.pkl
文件加载成功，共包含 100000 条记录。

--- 第一条记录结构检查 ---
  - ID: OEAV1298105
  - Days: 492
  - 包含的键数量: 4 (预期为 4)
  - 错误: 记录中缺少蛋白 'NSP1' 或 'aligned_NSP1' 的信息。
  - 错误: 记录中缺少蛋白 'NSP15' 或 'aligned_NSP15' 的信息。
  - 错误: 记录中缺少蛋白 'N' 或 'aligned_N' 的信息。


In [26]:
loaded_data[0]

{'id': 'OEAV1298105',
 'days': 492,
 'S': {'seq_t': array([ 0, 20, 18, ..., 19, 11,  2]),
  'structure_t': array([4098, 3425, 1339, ...,  490, 1272, 4097])},
 'aligned_S': {'seq_t': array([ 0, 20, 18, ..., 19, 11,  2]),
  'structure_t': array([4098, 3425, 1339, ...,  490, 1272, 4097])}}

In [32]:
with open('/data2/zhoukaitao/01evoModel/dataset/251020_review/7country/ESMC/USA.pkl', 'rb') as f:
    loaded_data = pickle.load(f)
    
loaded_data

[{'id': 'OEAV8174725',
  'days': 58,
  'S': array([ 0, 20, 18, ..., 19, 11,  2]),
  'aligned_S': array([ 0, 20, 18, ..., 19, 11,  2])},
 {'id': 'OEAV18943507',
  'days': 31,
  'S': array([ 0, 20, 18, ..., 19, 11,  2]),
  'aligned_S': array([ 0, 20, 18, ..., 19, 11,  2])},
 {'id': 'OEAV18942895',
  'days': 31,
  'S': array([ 0, 20, 18, ..., 19, 11,  2]),
  'aligned_S': array([ 0, 20, 18, ..., 19, 11,  2])},
 {'id': 'OEAV1094185',
  'days': 52,
  'S': array([ 0, 20, 18, ..., 19, 11,  2]),
  'aligned_S': array([ 0, 20, 18, ..., 19, 11,  2])},
 {'id': 'OEAV18942466',
  'days': 31,
  'S': array([ 0, 20, 18, ..., 19, 11,  2]),
  'aligned_S': array([ 0, 20, 18, ..., 19, 11,  2])},
 {'id': 'OEAV18943381',
  'days': 31,
  'S': array([ 0, 20, 18, ..., 19, 11,  2]),
  'aligned_S': array([ 0, 20, 18, ..., 19, 11,  2])},
 {'id': 'OEAV18942651',
  'days': 31,
  'S': array([ 0, 20, 18, ..., 19, 11,  2]),
  'aligned_S': array([ 0, 20, 18, ..., 19, 11,  2])},
 {'id': 'OEAV18942632',
  'days': 31,
  'S'